# Churn & Retention — ML

Цель анализа — не просто предсказать churn, а понять, **как ранжировать клиентов по риску и кого приоритизировать для retention**.

В ноутбуке:
- проверяем качество данных;
- сравниваем Logistic Regression и Random Forest;
- оцениваем ROC-AUC и PR-AUC;
- смотрим ROC/PR-кривые;
- анализируем threshold;
- переводим ограничение retention-команды в operating point;
- оцениваем top-risk сегменты;
- проверяем калибровку вероятностей;
- смотрим permutation feature importance;
- формулируем бизнес-выводы и ограничения.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
DATA_PATH = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Churn rate:", f"{df['Churn'].mean():.2%}")
display(df.isna().sum().sort_values(ascending=False).head(10))


## Распределение churn


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x="Churn", ax=ax)
ax.set_title("Распределение целевой переменной")
ax.set_xlabel("Churn")
ax.set_ylabel("Количество клиентов")
plt.tight_layout()


## Train / test и preprocessing

Используем stratified split. Все преобразования находятся внутри Pipeline, поэтому параметры imputation/scaling/encoding не обучаются на test.


In [ ]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"].astype(int)

categorical = X.select_dtypes(include=["object"]).columns.tolist()
numeric = X.select_dtypes(exclude=["object"]).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

fitted = {}
probas = {}
comparison = []

for name, estimator in models.items():
    model = Pipeline([
        ("preprocess", preprocessor),
        ("model", estimator),
    ])
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]

    fitted[name] = model
    probas[name] = p

    comparison.append({
        "model": name,
        "ROC-AUC": roc_auc_score(y_test, p),
        "PR-AUC": average_precision_score(y_test, p),
    })

comparison_df = pd.DataFrame(comparison).sort_values("PR-AUC", ascending=False)
display(comparison_df)


## ROC и PR-кривые

ROC-AUC полезна для общего разделения классов, а PR-AUC лучше отражает качество поиска меньшего класса churn.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, p in probas.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, p):.3f})")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_title("ROC curve")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, p in probas.items():
    precision, recall, _ = precision_recall_curve(y_test, p)
    ax.plot(recall, precision, label=f"{name} (AP={average_precision_score(y_test, p):.3f})")
ax.axhline(y_test.mean(), linestyle="--", label="Churn rate baseline")
ax.set_title("Precision-Recall curve")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
plt.tight_layout()


## Выбор модели

Для дальнейшего business analysis выбираем модель с максимальным PR-AUC на test. Это не означает, что PR-AUC автоматически является единственным критерием выбора в реальном продукте; здесь это прозрачное правило для проекта.


In [ ]:
best_name = comparison_df.iloc[0]["model"]
best_model = fitted[best_name]
best_proba = probas[best_name]

print("Selected model:", best_name)


## Threshold analysis

Threshold 0.5 не является автоматически правильным. Мы смотрим, как меняются размер retention-сегмента, precision, recall и доля пойманного churn.


In [ ]:
threshold_rows = []

for threshold in np.arange(0.20, 0.81, 0.05):
    pred = (best_proba >= threshold).astype(int)
    captured = ((pred == 1) & (y_test.to_numpy() == 1)).sum()
    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "customers_flagged": int(pred.sum()),
        "share_flagged": pred.mean(),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "share_of_churners_captured": captured / y_test.sum(),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_df["threshold"], threshold_df["precision"], marker="o", label="Precision")
ax.plot(threshold_df["threshold"], threshold_df["recall"], marker="o", label="Recall")
ax.plot(threshold_df["threshold"], threshold_df["f1"], marker="o", label="F1")
ax.set_title("Метрики в зависимости от threshold")
ax.set_xlabel("Threshold")
ax.set_ylabel("Metric")
ax.legend()
plt.tight_layout()


## Operating point от capacity retention-команды

Это более бизнесовый способ выбрать threshold.

Если команда может обработать только 5%, 10%, 20% или 30% клиентской базы, мы берём соответствующее количество самых рискованных клиентов и смотрим, сколько реального churn попало в этот сегмент.


In [ ]:
capacity_rows = []

order = np.argsort(-best_proba)
y_sorted = y_test.to_numpy()[order]
p_sorted = best_proba[order]

for capacity in [0.05, 0.10, 0.20, 0.30]:
    n = max(1, int(np.ceil(len(y_test) * capacity)))
    selected = y_sorted[:n]
    captured = int(selected.sum())

    capacity_rows.append({
        "capacity": capacity,
        "customers_contacted": n,
        "implied_threshold": p_sorted[n - 1],
        "precision": selected.mean(),
        "churners_captured": captured,
        "share_of_churners_captured": captured / y_test.sum(),
    })

capacity_df = pd.DataFrame(capacity_rows)
display(capacity_df.round(4))


## Top-risk coverage

Отдельно смотрим, какая доля всех наблюдаемых churn-клиентов находится среди top 5/10/20/30/50% клиентов по риску.


In [ ]:
top_rows = []

for share in [0.05, 0.10, 0.20, 0.30, 0.50]:
    n = max(1, int(np.ceil(len(y_test) * share)))
    selected = y_sorted[:n]
    captured = int(selected.sum())

    top_rows.append({
        "top_share": share,
        "customers": n,
        "precision": selected.mean(),
        "share_of_churners_captured": captured / y_test.sum(),
    })

top_risk_df = pd.DataFrame(top_rows)
display(top_risk_df.round(4))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(top_risk_df["top_share"] * 100, top_risk_df["share_of_churners_captured"] * 100, marker="o")
ax.set_title("Сколько churn покрывает top-risk сегмент")
ax.set_xlabel("Доля клиентской базы, которую обрабатываем (%)")
ax.set_ylabel("Доля churn, попавшая в сегмент (%)")
plt.tight_layout()


## Calibration

Для retention-задачи вероятность сама по себе является продуктом модели. Поэтому проверяем Brier score и сравниваем среднюю прогнозируемую вероятность с фактической частотой churn.


In [ ]:
brier = brier_score_loss(y_test, best_proba)

print("Brier score:", round(brier, 4))
print("Mean predicted probability:", round(best_proba.mean(), 4))
print("Observed churn rate:", round(y_test.mean(), 4))


In [ ]:
from sklearn.calibration import calibration_curve

fraction_pos, mean_pred = calibration_curve(y_test, best_proba, n_bins=10, strategy="quantile")

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(mean_pred, fraction_pos, marker="o", label=best_name)
ax.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
ax.set_title("Calibration curve")
ax.set_xlabel("Средняя предсказанная вероятность")
ax.set_ylabel("Фактическая доля churn")
ax.legend()
plt.tight_layout()


## Permutation feature importance

Importance считается на untouched test set с PR-AUC в качестве scoring metric. Это importance исходных customer-level признаков, а не отдельных one-hot колонок.

Важно: importance показывает связь признака с работой модели, но не причинность.


In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)

importance_df = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)

display(importance_df.head(15).round(5))

top = importance_df.head(10).sort_values("importance_mean")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top["feature"], top["importance_mean"])
ax.set_title("Top permutation feature importance")
ax.set_xlabel("Decrease in PR-AUC after permutation")
plt.tight_layout()


## Финальная интерпретация

Модель отвечает на вопрос **«кого считать более рискованным?»**, но не отвечает на вопрос **«какое действие удержит этого клиента?»**.

Поэтому следующий реальный шаг — retention experiment:

1. выбрать ограниченный top-risk сегмент;
2. разделить его на treatment и control;
3. предложить intervention только treatment-группе;
4. измерить фактическое снижение churn;
5. сравнить эффект с затратами на удержание.

Таким образом, ML используется как слой **приоритизации**, а эксперимент — как слой **проверки причинного эффекта**.


## Ограничения

- Датасет исторический.
- Test set используется для финальной оценки и feature importance, но не для обучения.
- Predictive risk не равен causal churn probability.
- Threshold зависит от стоимости контакта, capacity команды и стоимости потерянного клиента.
- Для production-модели понадобятся временная валидация, мониторинг drift и регулярная переоценка calibration.
